<a href="https://colab.research.google.com/github/azcsprof/ASU-CSE475-SS25/blob/Unit-2-Lab-2/Unit2_Lab2_Tutorial.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Step 1: Import Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn import metrics
from sklearn.metrics import roc_curve, auc

In [ ]:
# STEP 2: Load dataset from from GitHub
url = "https://raw.githubusercontent.com/azcsprof/ASU-CSE475-SS25/Unit-2-Lab-2/HeartDisease.csv"
HeartDisease = pd.read_csv(url)

In [ ]:
# Step 3: EDA – Overview
# --- GENERAL STRUCTURE + SUMMARY STATS ---

print("📌 First 5 rows:")
print(HeartDisease.head())

print("\n📌 Shape of the dataset:")
print("✅ shape_data:", HeartDisease.shape)

print("\n📌 Data types and non-null counts:")
print(HeartDisease.info())

print("\n📌 Summary statistics for numeric columns:")
print(HeartDisease.describe())

print("\n📌 Null values per column:")
print(HeartDisease.isnull().sum())

print("\n📌 Number of duplicate rows:")
print(HeartDisease.duplicated().sum())

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# Create a heatmap showing missing values
plt.figure(figsize=(10, 4))
sns.heatmap(HeartDisease.isnull(), cbar=False, cmap='viridis')
plt.title("📊 Missing Values Heatmap")
plt.show()

In [ ]:
# Step 4: Visual EDA
plt.figure(figsize=(10, 6))
sns.countplot(x='target', data=HeartDisease)
plt.title("Class Distribution (Heart Disease)")
plt.ylabel("Count")
plt.show()

In [ ]:
plt.figure(figsize=(12, 8))
sns.heatmap(HeartDisease.corr(), annot=True, cmap='coolwarm')
plt.title("Correlation Matrix")
plt.show()

In [ ]:
# Feature/Target Split
X = HeartDisease.drop(columns='target')
y = HeartDisease['target']

In [ ]:
# Step 6: Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=0)

In [ ]:
# Step 7: Train Random Forest
rf_clf = RandomForestClassifier(
    criterion="gini",
    min_samples_leaf=1,
    min_samples_split=10,
    max_features='sqrt',
    random_state=1
)
rf_clf.fit(X_train, y_train)
y_pred_rf = rf_clf.predict(X_test)

In [ ]:
# 📏 Step 8: Evaluate Random Forest

rf_accuracy = metrics.accuracy_score(y_test, y_pred_rf)
rf_precision = metrics.precision_score(y_test, y_pred_rf)
rf_recall = metrics.recall_score(y_test, y_pred_rf)
rf_f1 = metrics.f1_score(y_test, y_pred_rf)

print("🔹 Random Forest Performance 🔹")
print("Accuracy:", rf_accuracy)
print("Precision:", rf_precision)
print("Recall:", rf_recall)
print("F1 Score:", rf_f1)

# 🧠 Interpretation:
print("\nInterpretation Guide:")
print("These metrics help assess classification performance beyond just overall correctness:")
print("• Accuracy: Overall % of correct predictions. Can be misleading with imbalanced classes.")
print("• Precision: Out of all predicted positives, how many were actually positive? High precision = few false positives.")
print("• Recall: Out of all actual positives, how many did we correctly identify? High recall = few false negatives.")
print("• F1 Score: The harmonic mean of precision and recall. Useful when you need a balance between both.")
print("\nChoose your metric based on the problem domain:")
print("• Prioritize **precision** if false positives are costly (e.g., spam filters).")
print("• Prioritize **recall** if false negatives are dangerous (e.g., medical diagnosis).")
print("• Use **F1 Score** when both matter and class imbalance is a concern.")

In [ ]:
# Step 9: ROC and AUC – Random Forest

from sklearn.metrics import roc_curve, auc

# Get predicted probabilities for the positive class
rf_probs = rf_clf.predict_proba(X_test)[:, 1]

# Compute ROC curve and AUC score
rf_fpr, rf_tpr, _ = roc_curve(y_test, rf_probs)
rf_auc = auc(rf_fpr, rf_tpr)

# Plot ROC curve
plt.plot(rf_fpr, rf_tpr, label=f"Random Forest AUC = {rf_auc:.2f}", color='darkorange')
plt.plot([0, 1], [0, 1], 'b--')  # Baseline (random guessing)
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve – Random Forest")
plt.legend(loc='lower right')
plt.grid(True)
plt.show()

# Interpretation:
print("\nInterpretation:")
print("The ROC (Receiver Operating Characteristic) curve shows the trade-off between True Positive Rate and False Positive Rate.")
print("• The closer the curve hugs the top-left corner, the better the model is at separating the two classes.")
print("• The diagonal dashed line represents random guessing (AUC = 0.50).")
print(f"• This model's AUC is {rf_auc:.2f}, which indicates strong performance if it's above 0.80.")
print("AUC (Area Under the Curve) is useful for comparing classifiers, especially when class imbalance is present.")
print("Use ROC curves to choose classification thresholds and to evaluate general model quality independent of accuracy.")

In [ ]:
# Step 10: K-Fold Cross Validation – Random Forest

max_acc, max_k = 0, 0

for k in range(2, 11):
    skfold = StratifiedKFold(n_splits=k, shuffle=True, random_state=100)
    acc = cross_val_score(rf_clf, X, y, cv=skfold).mean() * 100

    if acc > max_acc:
        max_acc = acc
        max_k = k

    print(f"k = {k}, Accuracy = {acc:.2f}%")

best_accuracy = max_acc
best_k_fold = max_k

print("\nBest Accuracy:", best_accuracy)
print("Best k:", best_k_fold)

# Interpretation Guide:
print("\nInterpretation:")
print("K-Fold Cross Validation splits the data into 'k' subsets (folds) to assess model performance more robustly.")
print("Each fold takes a turn as the test set while the others form the training set, reducing the risk of overfitting to a single train-test split.")
print("This loop tests different values of k (from 2 to 10) to find the one that yields the highest average accuracy.")
print("• Higher k (e.g., 10) gives more stable estimates but increases computation.")
print("• Lower k (e.g., 2 or 3) is faster but may be noisier.")
print("\nThe 'Best k' value indicates the number of folds that provided the most reliable performance for the model.")
print("Use this result to set a consistent cross-validation strategy going forward.")

In [ ]:
# Train Logistic Regression
logreg = LogisticRegression(solver='liblinear', random_state=0)
logreg.fit(X_train, y_train)
y_pred_logreg = logreg.predict(X_test)

# Evaluate
log_accuracy = metrics.accuracy_score(y_test, y_pred_logreg)
log_precision = metrics.precision_score(y_test, y_pred_logreg)
log_recall = metrics.recall_score(y_test, y_pred_logreg)
log_f1 = metrics.f1_score(y_test, y_pred_logreg)

print("🔹 Logistic Regression Performance 🔹")
print("Accuracy:", log_accuracy)
print("Precision:", log_precision)
print("Recall:", log_recall)
print("F1 Score:", log_f1)

In [ ]:
# ROC and AUC – Logistic Regression
log_probs = logreg.predict_proba(X_test)[:, 1]
log_fpr, log_tpr, _ = roc_curve(y_test, log_probs)
log_auc = auc(log_fpr, log_tpr)

plt.plot(log_fpr, log_tpr, label=f"Logistic Regression AUC = {log_auc:.2f}", color='green')
plt.plot([0,1], [0,1], 'b--')
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve – Logistic Regression")
plt.legend(loc='lower right')
plt.grid(True)
plt.show()

In [ ]:
# Final Comparison Table

comparison_df = pd.DataFrame({
    "Model": ["Random Forest", "Logistic Regression"],
    "Accuracy": [rf_accuracy, log_accuracy],
    "Precision": [rf_precision, log_precision],
    "Recall": [rf_recall, log_recall],
    "F1 Score": [rf_f1, log_f1],
    "AUC": [rf_auc, log_auc]
})

# Display the table explicitly
print("\n Final Model Comparison Table:")
print(comparison_df.round(3))  # Rounds metrics to 3 decimal places for cleaner output

# Interpretation Guide:
print("\nInterpretation:")
print("This table summarizes the key performance metrics for two classification models on the same dataset.")
print("• Accuracy shows overall correctness, but may mislead with imbalanced classes.")
print("• Precision = TP / (TP + FP) → high means few false positives.")
print("• Recall = TP / (TP + FN) → high means few false negatives.")
print("• F1 Score balances precision and recall — useful when both matter.")
print("• AUC reflects how well the model separates the two classes across thresholds.")
print("\nUse this comparison to weigh trade-offs based on your application:")
print("• High precision → safer predictions, fewer false alarms.")
print("• High recall → better sensitivity, fewer missed detections.")
print("• High AUC → more confident discrimination between classes regardless of threshold.")

In [ ]:
from sklearn.svm import SVC

# Initialize an SVM classifier using a linear kernel
# probability=True enables probability estimates (needed for ROC/AUC)
svm_clf = SVC(kernel='linear', probability=True, random_state=0)

# Train the SVM model
svm_clf.fit(X_train, y_train)

# Make predictions on the test set
y_pred_svm = svm_clf.predict(X_test)

# Evaluate model performance using standard classification metrics
svm_accuracy = metrics.accuracy_score(y_test, y_pred_svm)
svm_precision = metrics.precision_score(y_test, y_pred_svm)
svm_recall = metrics.recall_score(y_test, y_pred_svm)
svm_f1 = metrics.f1_score(y_test, y_pred_svm)

# Display results
print("🔹 Support Vector Machine Performance 🔹")
print("Accuracy:", svm_accuracy)
print("Precision:", svm_precision)
print("Recall:", svm_recall)
print("F1 Score:", svm_f1)

print("\n Interpretation:")
print("SVMs work by finding the optimal hyperplane that maximally separates the two classes.")
print("You're using a **linear kernel**, so this model is assuming the classes can be separated with a straight line (or plane).")
print()
print("• Accuracy: The overall proportion of correct predictions.")
print("• Precision: Of all instances predicted as positive, how many were correct? Useful when false positives are costly.")
print("• Recall: Of all actual positives, how many were correctly identified? Critical when false negatives are risky.")
print("• F1 Score: A combined metric that balances precision and recall — useful for imbalanced datasets.")
print()
print("If SVM performs well across these metrics, it suggests that a linear boundary is sufficient.")
print("If recall is low, the SVM may be too conservative in flagging positive cases.")
print("If precision is low, it may be predicting too many false positives.")

In [ ]:
# ROC and AUC – SVM
svm_probs = svm_clf.predict_proba(X_test)[:, 1]
svm_fpr, svm_tpr, _ = roc_curve(y_test, svm_probs)
svm_auc = auc(svm_fpr, svm_tpr)

plt.plot(svm_fpr, svm_tpr, label=f"SVM AUC = {svm_auc:.2f}", color='purple')
plt.plot([0,1], [0,1], 'b--')
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve – Support Vector Machine")
plt.legend(loc='lower right')
plt.grid(True)
plt.show()

In [ ]:
# Updated Model Comparison Table
comparison_df = pd.DataFrame({
    "Model": ["Random Forest", "Logistic Regression", "SVM"],
    "Accuracy": [rf_accuracy, log_accuracy, svm_accuracy],
    "Precision": [rf_precision, log_precision, svm_precision],
    "Recall": [rf_recall, log_recall, svm_recall],
    "F1 Score": [rf_f1, log_f1, svm_f1],
    "AUC": [rf_auc, log_auc, svm_auc]
})

comparison_df

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn import metrics

# STEP: Standardize Features and Train Logistic Regression

# 1. Standardize features (mean = 0, std = 1)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# 2. Train-test split using standardized features
X_train_std, X_test_std, y_train_std, y_test_std = train_test_split(
    X_scaled, y, test_size=0.3, random_state=0
)

# 3. Train logistic regression model
log_clf_std = LogisticRegression()
log_clf_std.fit(X_train_std, y_train_std)

# 4. Predict and evaluate
y_pred_std = log_clf_std.predict(X_test_std)

log_accuracy = metrics.accuracy_score(y_test_std, y_pred_std)
log_precision = metrics.precision_score(y_test_std, y_pred_std)
log_recall = metrics.recall_score(y_test_std, y_pred_std)
log_f1 = metrics.f1_score(y_test_std, y_pred_std)

print("🔹 Logistic Regression (Standardized) Performance 🔹")
print("Accuracy:", log_accuracy)
print("Precision:", log_precision)
print("Recall:", log_recall)
print("F1 Score:", log_f1)

# Interpretation Guide:
print("\nInterpretation:")
print("Standardization ensures that all input features contribute equally to the logistic regression model.")
print("This improves training stability and model interpretability.")
print("• Accuracy shows overall correctness.")
print("• Precision shows how many predicted positives were actually correct.")
print("• Recall shows how many true positives were correctly identified.")
print("• F1 balances precision and recall — helpful when you care about both.")
print("By standardizing, we make logistic regression fairer and more effective across varied feature scales.")

In [ ]:
# Step: Train Logistic Regression on Standardized Features

# Import logistic regression model from scikit-learn
from sklearn.linear_model import LogisticRegression

# Create a logistic regression model object
# - 'liblinear' is a solver suitable for small datasets and supports L1/L2 regularization
# - 'random_state=0' ensures consistent results every time the model is trained
logreg_std = LogisticRegression(solver='liblinear', random_state=0)

# Fit the logistic regression model to the standardized training data
# - X_train_std: standardized predictor variables
# - y_train_std: corresponding class labels (e.g., 0 or 1 for binary classification)
# The model will learn the weights (coefficients) that best separate the two classes
logreg_std.fit(X_train_std, y_train_std)

In [ ]:
# Step: Create a Table of Standardized Coefficients for Logistic Regression

# Build a readable DataFrame to display feature names and their corresponding standardized coefficients (beta weights)
# - 'X.columns' contains the names of the original features (in order)
# - 'logreg_std.coef_[0]' contains the learned weights from the trained logistic regression model
#   (Note: .coef_ is a 2D array; we use [0] to extract the coefficients for the first—and only—class in binary classification)

coefficients = pd.DataFrame({
    "Feature": X.columns,
    "Standardized Coefficient (Beta)": logreg_std.coef_[0]
})

# Sort by absolute value of the coefficients to see which features have the strongest influence
coefficients = coefficients.sort_values(
    by="Standardized Coefficient (Beta)",
    key=abs,
    ascending=False
)

# Display the result
coefficients

In [ ]:
# Plot the standardized coefficients from logistic regression
plt.figure(figsize=(10, 6))

# Create a horizontal barplot showing the strength and direction of each feature's effect
sns.barplot(data=coefficients, x="Standardized Coefficient (Beta)", y="Feature")

# Add a vertical reference line at 0 for visual separation
plt.axvline(0, color='gray', linestyle='--')

# Set plot title and formatting
plt.title("Logistic Regression: Standardized Coefficients")
plt.grid(True)
plt.tight_layout()
plt.show()

# 🔍 Interpretation Guidance:
# - Features with **positive coefficients** increase the likelihood of the positive class (label = 1).
# - Features with **negative coefficients** reduce the likelihood of the positive class.
# - The **magnitude** (absolute value) of each coefficient reflects the **relative importance** of that feature,
#   assuming all features were standardized (mean = 0, std = 1).
# - This plot helps you understand which predictors drive the model's classification decisions the most,
#   and in what direction they push the prediction.

In [ ]:
# Confusion Matrix for Random Forest
from sklearn.metrics import confusion_matrix  # Import function to create confusion matrix

# Compute the confusion matrix using actual vs predicted values
rf_cm = confusion_matrix(y_test, y_pred_rf)

# Visualize the confusion matrix as a heatmap
sns.heatmap(rf_cm, annot=True, fmt='d', cmap='Blues')
plt.title("Random Forest – Confusion Matrix")
plt.xlabel("Predicted Label")
plt.ylabel("Actual Label")
plt.show()

# Interpretation Guidance:
# - The top-left cell shows the number of **true negatives** (correctly predicted class 0).
# - The bottom-right cell shows the number of **true positives** (correctly predicted class 1).
# - The top-right cell shows **false positives** (predicted 1 but actual is 0).
# - The bottom-left cell shows **false negatives** (predicted 0 but actual is 1).
# - A strong model has most of its counts on the diagonal (top-left and bottom-right),
#   meaning it’s making correct predictions.

In [ ]:
# Confusion Matrix for Logistic Regression
log_cm = confusion_matrix(y_test, y_pred_logreg)
sns.heatmap(log_cm, annot=True, fmt='d', cmap='Greens')
plt.title("Logistic Regression – Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()

In [ ]:
# Confusion Matrix for SVM
svm_cm = confusion_matrix(y_test, y_pred_svm)
sns.heatmap(svm_cm, annot=True, fmt='d', cmap='Purples')
plt.title("Support Vector Machine – Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()

In [ ]:
# Extract counts from confusion matrix
rf_tp = rf_cm[1, 1]  # True Positives: predicted 1, actual 1
rf_fp = rf_cm[0, 1]  # False Positives: predicted 1, actual 0
rf_fn = rf_cm[1, 0]  # False Negatives: predicted 0, actual 1
rf_tn = rf_cm[0, 0]  # True Negatives: predicted 0, actual 0

# Display performance metrics
print("Random Forest Metrics:")
print("Accuracy:", metrics.accuracy_score(y_test, y_pred_rf))  # Overall correctness
print("Precision:", rf_tp / (rf_tp + rf_fp))  # Of the predicted positives, how many were correct?
print("Recall:", rf_tp / (rf_tp + rf_fn))     # Of all actual positives, how many were identified?
print("F1 Score:", 2 * (rf_tp / (rf_tp + rf_fp)) * (rf_tp / (rf_tp + rf_fn)) /
      ((rf_tp / (rf_tp + rf_fp)) + (rf_tp / (rf_tp + rf_fn))))  # Harmonic mean of precision and recall

# 📌 Interpretation Guidance:
# - Accuracy tells you how often the classifier is correct overall.
# - Precision focuses on the quality of positive predictions: How often is a positive prediction correct?
# - Recall (sensitivity) tells you how many actual positive cases were caught.
# - F1 Score balances precision and recall—useful when there's class imbalance or trade-offs.

In [ ]:
metrics_table = pd.DataFrame({
    "Model": ["Random Forest", "Logistic Regression", "SVM"],
    "Accuracy": [rf_accuracy, log_accuracy, svm_accuracy],
    "Precision": [rf_precision, log_precision, svm_precision],
    "Recall": [rf_recall, log_recall, svm_recall],
    "F1 Score": [rf_f1, log_f1, svm_f1]
})

metrics_table

In [ ]:
# 🔍 Model Evaluation Helper Function
# This function evaluates a classification model by:
# 1. Computing and displaying the confusion matrix as a heatmap.
# 2. Printing key performance metrics: Accuracy, Precision, Recall, and F1 Score.
# 3. Returning all metrics so they can be stored, compared, or visualized later.
# Use this function to consistently evaluate different classifiers (e.g., Logistic Regression, Random Forest, SVM).

def evaluate_model(name, y_true, y_pred, cmap="Blues"):
    from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score
    cm = confusion_matrix(y_true, y_pred)

    # Confusion Matrix Heatmap
    plt.figure(figsize=(5, 4))
    sns.heatmap(cm, annot=True, fmt='d', cmap=cmap, cbar=False)
    plt.title(f"{name} – Confusion Matrix")
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.show()

    # Metrics
    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred)
    rec = recall_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred)

    print(f"🔍 {name} Metrics")
    print(f"Accuracy : {acc:.3f}")
    print(f"Precision: {prec:.3f}")
    print(f"Recall   : {rec:.3f}")
    print(f"F1 Score : {f1:.3f}")
    print("-" * 30)

    return acc, prec, rec, f1

In [ ]:
# ✅ Run model evaluation and interpretation for each classifier
# Each call prints a confusion matrix and four key metrics:
# Accuracy: Overall correct predictions
# Precision: How many predicted positives were actually positive
# Recall: How many actual positives were correctly identified
# F1 Score: Harmonic mean of precision and recall (good for imbalance)

# 🔍 Evaluate Random Forest
rf_metrics = evaluate_model("Random Forest", y_test, y_pred_rf, cmap="Blues")
# Interpretation Tip:
# Look for high recall if catching all positives is important.
# Look for high precision if avoiding false positives is more critical.

# 🔍 Evaluate Logistic Regression
log_metrics = evaluate_model("Logistic Regression", y_test, y_pred_logreg, cmap="Greens")
# Interpretation Tip:
# Logistic Regression is interpretable and often good when features are linearly separable.

# 🔍 Evaluate Support Vector Machine
svm_metrics = evaluate_model("Support Vector Machine", y_test, y_pred_svm, cmap="Purples")
# Interpretation Tip:
# SVMs tend to perform well with clean boundaries and fewer outliers, especially with linear kernels.